In [1]:
#Imports
import torch
import csv
from matplotlib import pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import random

#Data loading
with open("data/mnist_train.csv", "r") as read_obj:
    csv_reader = csv.reader(read_obj)
    train = list(csv_reader)
train = train[1:]
train = [[int(train[i][j]) for j in range(len(train[0]))] for i in range(len(train))]
train = torch.tensor(train, dtype = torch.float32)

#Data formatting and data loader
train_Y = train[:,0].long()
train_X = train[:,1:]

dataset = TensorDataset(train_X, train_Y)

batch_size = 64
dataloader = DataLoader(dataset, batch_size = batch_size, shuffle = True)

In [2]:
#Model declarations
#Encoder model
class Encoder(nn.Module):
    def __init__(self, latent_dim = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim)
        )

    def forward(self, x):
        return self.net(x)
    
#Predictor model
#TODO add extra variables for additional context for predictor model
class Predictor(nn.Module):
    def __init__(self, latent_dim = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, latent_dim)
        )
    
    def forward(self, x):
        return self.net(x)

#JEPA wrapper module
class MNIST_JEPA(nn.Module):
    def __init__(self, latent_dim = 128, ema = 0.99, mask_size = 14):
        super().__init__()
        self.context_encoder = Encoder(latent_dim)
        self.target_encoder = Encoder(latent_dim)
        self.predictor = Predictor(latent_dim)
        self.ema = ema
        self.mask_size = mask_size

        #update target encoders weights to be identical to context encoder
        self.update_target_encoder(momentum = 0)

        #target encoder will inherit context encoder weights
        #so no need to track gradients
        for param in self.target_encoder.parameters():
            param.requires_grad = False
        
    #updates target encoder weights
    @torch.no_grad()
    def update_target_encoder(self, momentum = None):
        m = self.ema if momentum is None else momentum
        for param_context, param_target in zip(self.context_encoder.parameters(), self.target_encoder.parameters()):
            param_target.data = m*param_target + (1-m)*param_context
    
    def apply_mask(self, x):
        x = x.clone()
        mask_size = self.mask_size
        x = x.view((-1,28,28))
        vert = random.randint(0, mask_size)
        hor = random.randint(0, mask_size)
        x[:,vert: vert + mask_size, hor: hor + mask_size] = 0
        return x.view((-1,28*28)), (vert, hor)

    def forward(self, x):
        x_masked, action = self.apply_mask(x)
        x_target = x

        z_context = self.context_encoder(x_masked)

        with torch.no_grad():
            z_target = self.target_encoder(x_target)
        
        z_pred = self.predictor(z_context)

        return z_pred, z_target.detach()



In [3]:
#Training setup
device = torch.device("cpu")

model = MNIST_JEPA(latent_dim = 128, ema = 0.999, mask_size = 10).to(device)
lr = 1e-4
optimizer = optim.Adam(list(model.context_encoder.parameters()) + list(model.predictor.parameters()), lr = lr)
loss_fn = nn.MSELoss()

#Training
print("Beginning model training . . .")
model.train()
for epoch in range(2):
    total_loss = 0
    for data_X, data_Y in dataloader:

        optimizer.zero_grad()
        
        #forward pass
        z_pred, z_target = model(data_X)

        #loss
        loss = loss_fn(z_pred, z_target)

        loss.backward()
        optimizer.step()

        model.update_target_encoder()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Average Loss: {total_loss/len(dataloader):.4f}")



Beginning model training . . .
Epoch 1 | Average Loss: 17.5716
Epoch 2 | Average Loss: 28.6182


In [4]:
#Append additional layer to see if the model can be used for digition
class VisionModel(nn.Module):
    def __init__(self, encoder_model, latent_dim = 128, target_dim = 10):
        super().__init__()
        for param in encoder_model.parameters():
            param.requires_grad = False
        
        self.encoder_model = encoder_model

        self.lin = nn.Linear(latent_dim, target_dim)

    def forward(self, x):
        x = self.encoder_model(x)
        x = self.lin(x)
        return x  

In [5]:
#Declaration and training of vision model for 
#downstream labeling
vision_model = VisionModel(model.target_encoder, latent_dim=128)
vision_optimizer = optim.Adam(vision_model.lin.parameters(), lr = lr)
vision_loss_fn = nn.CrossEntropyLoss()


vision_model.train()
for epoch in range(10):
    total_loss = 0
    running_acc = 0.0
    for data_X, data_Y in dataloader:
        vision_optimizer.zero_grad()

        pred = vision_model(data_X)

        loss = vision_loss_fn(pred, data_Y)

        with torch.no_grad():
            preds = torch.argmax(pred, dim = 1)
            correct = (preds == data_Y)
            accuracy = correct.float().sum()
            running_acc += accuracy

        loss.backward()
        vision_optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Average Loss: {total_loss/len(dataloader):.4f}, Acc: {running_acc/60000:.4f}")

Epoch 1 | Average Loss: 4.6276, Acc: 0.5092
Epoch 2 | Average Loss: 1.2113, Acc: 0.7383
Epoch 3 | Average Loss: 0.8913, Acc: 0.7810
Epoch 4 | Average Loss: 0.7464, Acc: 0.8018
Epoch 5 | Average Loss: 0.6639, Acc: 0.8156
Epoch 6 | Average Loss: 0.6092, Acc: 0.8259
Epoch 7 | Average Loss: 0.5719, Acc: 0.8346
Epoch 8 | Average Loss: 0.5427, Acc: 0.8396
Epoch 9 | Average Loss: 0.5188, Acc: 0.8460
Epoch 10 | Average Loss: 0.5003, Acc: 0.8506


In [ ]:
#Running PCA on the latent space to see if we have ten clusters
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import numpy as np

scaler = StandardScaler()
model.eval()
with torch.no_grad():
    X_latent, _ = model(train_X)
print(X_latent.shape)
X_latent_scaled = scaler.fit_transform(X_latent)

pca = PCA(n_components=10)
X_pca = pca.fit_transform(X_latent_scaled)

#Shamelessly copied from Gemini
explained_variance = pca.explained_variance_ratio_
for i, var in enumerate(explained_variance):
    print(f"Variance explained by PC{i+1}: {explained_variance[i]*100:.2f}%")

print(f"Total variance captured in 2D: {np.sum(explained_variance)*100:.2f}%")


torch.Size([60000, 128])
Variance explained by PC1: 25.44%
Variance explained by PC2: 13.48%
Variance explained by PC3: 11.25%
Variance explained by PC4: 8.35%
Variance explained by PC5: 7.89%
Variance explained by PC6: 5.72%
Variance explained by PC7: 5.30%
Variance explained by PC8: 4.72%
Variance explained by PC9: 3.67%
Variance explained by PC10: 2.43%
Total variance captured in 2D: 88.26%


" plt.figure(figsize=(10, 8))\nscatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.6, s=5)\nplt.colorbar(scatter, label='Digit Class')\nplt.title('2D PCA Embedding of MNIST Digits')\nplt.xlabel(f'Principal Component 1 ({explained_variance[0]*100:.1f}%)')\nplt.ylabel(f'Principal Component 2 ({explained_variance[1]*100:.1f}%)')\nplt.grid(True, linestyle='--', alpha=0.5)\nplt.show() "